# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_multicolor


In [1]:
from pathlib import Path
import json, os, zipfile, collections
import numpy as np
import pandas as pd

Competition='/kaggle/input/competitions/neurogolf-2026'
ROOT = Path.cwd()
TASK_ID = 'task349'
DATA_DIR = ROOT
OUT_DIR = ROOT / 'task349_rect_rule_generalized'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('TASK_ID =', TASK_ID)
print('ROOT =', ROOT)
print('OUT_DIR =', OUT_DIR)

TASK_ID = task349
ROOT = /kaggle/working
OUT_DIR = /kaggle/working/task349_rect_rule_generalized


In [2]:
# ONNX dependency setup for model export and validation.
import importlib.util, subprocess, sys
missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import onnx
import onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 75.0 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [3]:
# Load task JSON.
TASK_PATH = DATA_DIR / f'{TASK_ID}.json'
if not TASK_PATH.exists():
    TASK_PATH = Path(Competition, f'{TASK_ID}.json')

with open(TASK_PATH, 'r', encoding='utf-8') as f:
    task = json.load(f)

print('loaded:', TASK_PATH)
print({split: len(task.get(split, [])) for split in ['train', 'test', 'arc-gen']})

loaded: /kaggle/input/competitions/neurogolf-2026/task349.json
{'train': 4, 'test': 1, 'arc-gen': 262}


In [4]:
# Inspect example shapes and observed color-9 rectangle sizes.
H = W = 30
CH = 10

def all_examples(task):
    return task.get('train', []) + task.get('test', []) + task.get('arc-gen', [])

def find_9_components(grid):
    A = np.array(grid, dtype=np.int64)
    h, w = A.shape
    seen = np.zeros((h, w), dtype=bool)
    comps = []
    for r in range(h):
        for c in range(w):
            if A[r, c] == 9 and not seen[r, c]:
                stack = [(r, c)]
                seen[r, c] = True
                pts = []
                while stack:
                    rr, cc = stack.pop()
                    pts.append((rr, cc))
                    for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
                        nr, nc = rr + dr, cc + dc
                        if 0 <= nr < h and 0 <= nc < w and A[nr, nc] == 9 and not seen[nr, nc]:
                            seen[nr, nc] = True
                            stack.append((nr, nc))
                rs = [p[0] for p in pts]
                cs = [p[1] for p in pts]
                comps.append((min(rs), max(rs), min(cs), max(cs), len(pts)))
    return comps

shape_rows = []
rect_shapes = collections.Counter()
for split in ['train', 'test', 'arc-gen']:
    for idx, ex in enumerate(task.get(split, [])):
        inp = ex['input']
        shape_rows.append({'split': split, 'idx': idx, 'shape': (len(inp), len(inp[0]))})
        for r1, r2, c1, c2, area in find_9_components(inp):
            rh, rw = r2 - r1 + 1, c2 - c1 + 1
            rect_shapes[(rh, rw)] += 1

display(pd.DataFrame(shape_rows).groupby(['split', 'shape']).size().reset_index(name='count'))
print('observed color-9 rectangle shapes:', sorted(rect_shapes.items()))

,split,shape,count
0,arc-gen,"(10, 10)",59
1,arc-gen,"(15, 15)",50
2,arc-gen,"(20, 20)",57
3,arc-gen,"(25, 25)",42
4,arc-gen,"(30, 30)",54
5,test,"(30, 30)",1
6,train,"(10, 10)",1
7,train,"(15, 15)",1
8,train,"(20, 20)",2


observed color-9 rectangle shapes: [((1, 2), 20), ((2, 2), 242), ((2, 4), 7), ((2, 6), 3), ((2, 8), 1), ((2, 10), 2), ((3, 4), 21), ((3, 6), 4), ((3, 8), 4), ((4, 4), 118), ((4, 6), 6), ((4, 8), 1), ((5, 6), 12), ((5, 8), 2), ((5, 10), 1), ((6, 6), 61), ((6, 8), 4), ((6, 10), 2), ((7, 8), 7), ((8, 8), 27), ((9, 10), 1), ((10, 10), 12)]


## Task-specific model construction

This version uses a fully deterministic ONNX graph.  The model does not fit a tree, does not memorize visible input/output templates, and does not use `Loop`, `Scan`, `NonZero`, `Unique`, `Script`, or `Function`.

To improve hidden generalization, it detects **all solid color-9 rectangles with height and width from 1 to 13**, not just the rectangle shapes observed in the uploaded data.

In [5]:
# Manual ONNX graph builder for task349 rectangle morphology.
# Static input/output: [1, 10, 30, 30] one-hot tensor.

H = W = 30
CH = 10
FORBIDDEN = {'Loop', 'Scan', 'NonZero', 'Unique', 'Script', 'Function'}
SHAPE_LIMIT = 13  # all h,w in 1..13; keeps model < 1.4MB while broader than visible shapes.

class GraphBuilder:
    def __init__(self):
        self.nodes = []
        self.initializers = []
        self.counter = 0

    def name(self, prefix):
        self.counter += 1
        return f'{prefix}_{self.counter}'

    def init_array(self, name, arr):
        self.initializers.append(numpy_helper.from_array(np.asarray(arr), name))
        return name

    def const(self, value, dtype=np.float32):
        name = self.name('const')
        self.init_array(name, np.asarray(value, dtype=dtype))
        return name

    def slice(self, x, starts, ends, axes, prefix='slice'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node(
            'Slice',
            [x, self.const(starts, np.int64), self.const(ends, np.int64), self.const(axes, np.int64)],
            [y],
            name=y,
        ))
        return y

    def cast(self, x, to=TensorProto.FLOAT, prefix='cast'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node('Cast', [x], [y], to=to, name=y))
        return y

    def conv(self, x, weight, prefix='conv'):
        y = self.name(prefix)
        w_name = self.name('W')
        self.init_array(w_name, weight.astype(np.float32))
        self.nodes.append(helper.make_node('Conv', [x, w_name], [y], name=y))
        return y

    def conv_transpose(self, x, weight, prefix='deconv'):
        y = self.name(prefix)
        w_name = self.name('WT')
        self.init_array(w_name, weight.astype(np.float32))
        self.nodes.append(helper.make_node('ConvTranspose', [x, w_name], [y], name=y))
        return y

    def pad(self, x, pads, prefix='pad'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node(
            'Pad',
            [x, self.const(pads, np.int64), self.const([0.0], np.float32)],
            [y],
            mode='constant',
            name=y,
        ))
        return y

    def greater(self, x, threshold, prefix='gt'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node('Greater', [x, self.const([threshold], np.float32)], [y], name=y))
        return y

    def less(self, x, threshold, prefix='lt'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node('Less', [x, self.const([threshold], np.float32)], [y], name=y))
        return y

    def mul(self, a, b, prefix='mul'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node('Mul', [a, b], [y], name=y))
        return y

    def add(self, a, b, prefix='add'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node('Add', [a, b], [y], name=y))
        return y

    def sub(self, a, b, prefix='sub'):
        y = self.name(prefix)
        self.nodes.append(helper.make_node('Sub', [a, b], [y], name=y))
        return y


def export_task349_rect_rule(model_path, shape_limit=SHAPE_LIMIT):
    b = GraphBuilder()

    zero = b.slice('input', [0], [1], [1], prefix='zero_ch')
    nine = b.slice('input', [9], [10], [1], prefix='nine_ch')
    padded_nine = b.pad(nine, [0, 0, 1, 1, 0, 0, 1, 1], prefix='pad_nine')

    zero_acc = b.mul(zero, b.const([0.0], np.float32), prefix='zero_acc')
    frame_sum = zero_acc
    shadow_sum = zero_acc

    rectangle_shapes = [(h, w) for h in range(1, shape_limit + 1) for w in range(1, shape_limit + 1)]

    for h, w in rectangle_shapes:
        # Candidate top-left locations where an h x w block is entirely color 9.
        inside = b.conv(nine, np.ones((1, 1, h, w), dtype=np.float32), prefix=f'inside_{h}_{w}')
        top_left = b.cast(b.greater(inside, float(h * w) - 0.5, prefix=f'full_{h}_{w}'), prefix='full_f')

        # Suppress sub-rectangles by requiring no adjacent color-9 immediately above/below/left/right.
        nrows = H - h + 1
        ncols = W - w + 1

        above_full = b.conv(padded_nine, np.ones((1, 1, 1, w), dtype=np.float32), prefix='above')
        above = b.slice(above_full, [0, 1], [nrows, W - w + 2], [2, 3], prefix='above_slice')

        below_full = b.conv(padded_nine, np.ones((1, 1, 1, w), dtype=np.float32), prefix='below')
        below = b.slice(below_full, [h + 1, 1], [h + 1 + nrows, W - w + 2], [2, 3], prefix='below_slice')

        left_full = b.conv(padded_nine, np.ones((1, 1, h, 1), dtype=np.float32), prefix='left')
        left = b.slice(left_full, [1, 0], [H - h + 2, ncols], [2, 3], prefix='left_slice')

        right_full = b.conv(padded_nine, np.ones((1, 1, h, 1), dtype=np.float32), prefix='right')
        right = b.slice(right_full, [1, w + 1], [H - h + 2, w + 1 + ncols], [2, 3], prefix='right_slice')

        for neighbor in [above, below, left, right]:
            empty = b.cast(b.less(neighbor, 0.5, prefix='neighbor_empty'), prefix='empty_f')
            top_left = b.mul(top_left, empty, prefix='exact_tl')

        # Rectangle rule: margin is ceil(max(h,w)/2).
        margin = (max(h, w) + 1) // 2
        padded_tl = b.pad(top_left, [0, 0, margin, margin, 0, 0, margin, margin], prefix='pad_tl')

        frame_h = h + 2 * margin
        frame_w = w + 2 * margin
        frame_kernel = np.ones((1, 1, frame_h, frame_w), dtype=np.float32)
        frame_kernel[:, :, margin:margin + h, margin:margin + w] = 0.0
        frame_full = b.conv_transpose(padded_tl, frame_kernel, prefix='frame_deconv')
        frame = b.slice(frame_full, [2 * margin, 2 * margin], [2 * margin + H, 2 * margin + W], [2, 3], prefix='frame_crop')
        frame_sum = b.add(frame_sum, frame, prefix='frame_add')

        # Blue shadow: original rectangle columns, starting after the green frame below the object.
        shadow_kernel = np.zeros((1, 1, H + h + 2 * margin, frame_w), dtype=np.float32)
        shadow_kernel[:, :, h + margin:h + margin + H, margin:margin + w] = 1.0
        shadow_full = b.conv_transpose(padded_tl, shadow_kernel, prefix='shadow_deconv')
        shadow = b.slice(shadow_full, [2 * margin, 2 * margin], [2 * margin + H, 2 * margin + W], [2, 3], prefix='shadow_crop')
        shadow_sum = b.add(shadow_sum, shadow, prefix='shadow_add')

    frame_mask = b.mul(b.cast(b.greater(frame_sum, 0.5, prefix='frame_any'), prefix='frame_f'), zero, prefix='frame_zero_only')
    one = b.const([1.0], np.float32)
    inv_frame = b.sub(one, frame_mask, prefix='inv_frame')
    shadow_mask = b.mul(
        b.mul(b.cast(b.greater(shadow_sum, 0.5, prefix='shadow_any'), prefix='shadow_f'), zero, prefix='shadow_zero_only'),
        inv_frame,
        prefix='shadow_not_frame',
    )
    inv_shadow = b.sub(one, shadow_mask, prefix='inv_shadow')

    ch0 = b.mul(b.mul(zero, inv_frame, prefix='zero_no_frame'), inv_shadow, prefix='zero_no_shadow')
    zlike = b.mul(zero, b.const([0.0], np.float32), prefix='empty_channel')

    output_channels = []
    for k in range(CH):
        if k == 0:
            output_channels.append(ch0)
        elif k == 1:
            output_channels.append(shadow_mask)
        elif k == 3:
            output_channels.append(frame_mask)
        elif k == 9:
            output_channels.append(nine)
        else:
            output_channels.append(zlike)

    b.nodes.append(helper.make_node('Concat', output_channels, ['output'], axis=1, name='output_concat'))

    graph = helper.make_graph(
        b.nodes,
        'task349_generalized_rectangle_morphology',
        [helper.make_tensor_value_info('input', TensorProto.FLOAT, [1, CH, H, W])],
        [helper.make_tensor_value_info('output', TensorProto.FLOAT, [1, CH, H, W])],
        initializer=b.initializers,
    )
    model = helper.make_model(graph, opset_imports=[helper.make_operatorsetid('', 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    onnx.save(model, str(model_path))

    return {
        'model_type': 'deterministic_rectangular_morphology',
        'shape_limit': shape_limit,
        'enumerated_rectangle_shapes': len(rectangle_shapes),
        'rectangle_shapes': rectangle_shapes,
        'margin_rule': 'ceil(max(height,width)/2)',
    }

model_path = OUT_DIR / f'{TASK_ID}.onnx'
build_info = export_task349_rect_rule(model_path)
print('saved model:', model_path)
print('file_size_bytes:', model_path.stat().st_size)
print('build_info:', {k: v for k, v in build_info.items() if k != 'rectangle_shapes'})

saved model: /kaggle/working/task349_rect_rule_generalized/task349.onnx
file_size_bytes: 1350724
build_info: {'model_type': 'deterministic_rectangular_morphology', 'shape_limit': 13, 'enumerated_rectangle_shapes': 169, 'margin_rule': 'ceil(max(height,width)/2)'}


In [6]:
# Visible validation through ONNX Runtime.
# Single-thread settings avoid expensive/hanging thread-pool behavior for this static Conv-heavy graph.

def grid_to_tensor(grid):
    arr = np.zeros((1, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            arr[0, int(value), r, c] = 1.0
    return arr

def validate_onnx(model_path, task):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1
    opts.inter_op_num_threads = 1
    opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
    sess = ort.InferenceSession(str(model_path), sess_options=opts, providers=['CPUExecutionProvider'])

    split_rows = []
    first_wrong = None
    total_right = 0
    total = 0
    for split in ['train', 'test', 'arc-gen']:
        right = 0
        count = 0
        for idx, ex in enumerate(task.get(split, [])):
            pred = sess.run(['output'], {'input': grid_to_tensor(ex['input'])})[0]
            expected = grid_to_tensor(ex['output'])
            pred_binary = (pred > 0.5).astype(np.float32)
            ok = np.array_equal(pred_binary, expected)
            right += int(ok)
            count += 1
            total_right += int(ok)
            total += 1
            if not ok and first_wrong is None:
                first_wrong = {
                    'split': split,
                    'idx': idx,
                    'different_tensor_entries': int(np.sum(pred_binary != expected)),
                }
        split_rows.append({'split': split, 'right': right, 'total': count})
    return total_right, total, first_wrong, split_rows

right, total, first_wrong, split_rows = validate_onnx(model_path, task)
validation_row = {
    'task_id': TASK_ID,
    'right': right,
    'total': total,
    'accuracy': right / total if total else None,
    'first_wrong': first_wrong,
}
for row in split_rows:
    validation_row[f"{row['split']}_right"] = row['right']
    validation_row[f"{row['split']}_total"] = row['total']

print(validation_row)
assert right == total, validation_row

{'task_id': 'task349', 'right': 267, 'total': 267, 'accuracy': 1.0, 'first_wrong': None, 'train_right': 4, 'train_total': 4, 'test_right': 1, 'test_total': 1, 'arc-gen_right': 262, 'arc-gen_total': 262}


In [7]:
# ONNX architecture and forbidden-op checks.
model = onnx.load(str(model_path))
ops = collections.Counter(node.op_type for node in model.graph.node)
forbidden_present = sorted(FORBIDDEN & set(ops))
size_bytes = model_path.stat().st_size
architecture_report = {
    'task_id': TASK_ID,
    'file_size_bytes': size_bytes,
    'under_1_4mb': size_bytes < 1_400_000,
    'forbidden_ops_present': forbidden_present,
    'op_counts': dict(sorted(ops.items())),
}
print(architecture_report)
assert size_bytes < 1_400_000, size_bytes
assert not forbidden_present, forbidden_present

{'task_id': 'task349', 'file_size_bytes': 1350724, 'under_1_4mb': True, 'forbidden_ops_present': [], 'op_counts': {'Add': 338, 'Cast': 847, 'Concat': 1, 'Conv': 845, 'ConvTranspose': 338, 'Greater': 171, 'Less': 676, 'Mul': 683, 'Pad': 170, 'Slice': 1016, 'Sub': 2}}


In [8]:
# Persist verification/profile/manifest and build Kaggle-style submission.zip.
profile_df = pd.DataFrame([{
    'task_id': TASK_ID,
    'model_path': str(model_path),
    'model_version': 'task349-rect-rule-all-hw-le13',
    'file_size_bytes': model_path.stat().st_size,
    'under_1_4mb': model_path.stat().st_size < 1_400_000,
    'forbidden_ops_present': ','.join(architecture_report['forbidden_ops_present']),
    'visible_right': validation_row['right'],
    'visible_total': validation_row['total'],
    'train_right': validation_row['train_right'],
    'train_total': validation_row['train_total'],
    'test_right': validation_row['test_right'],
    'test_total': validation_row['test_total'],
    'arc_gen_right': validation_row['arc-gen_right'],
    'arc_gen_total': validation_row['arc-gen_total'],
}])
profile_path = OUT_DIR / 'profile.csv'
verification_path = OUT_DIR / 'verification_task349_rect_rule.csv'
profile_df.to_csv(profile_path, index=False)
profile_df.to_csv(verification_path, index=False)

manifest = {
    'task_id': TASK_ID,
    'model_version': 'task349-rect-rule-all-hw-le13',
    'model_path': str(model_path),
    'build_info': {k: v for k, v in build_info.items() if k != 'rectangle_shapes'},
    'validation': validation_row,
    'architecture_report': architecture_report,
}
manifest_path = OUT_DIR / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

submission_zip = ROOT / 'submission.zip'
with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(model_path, model_path.name)

print('profile:', profile_path)
print('verification:', verification_path)
print('manifest:', manifest_path)
print('submission:', submission_zip)

profile: /kaggle/working/task349_rect_rule_generalized/profile.csv
verification: /kaggle/working/task349_rect_rule_generalized/verification_task349_rect_rule.csv
manifest: /kaggle/working/task349_rect_rule_generalized/manifest.json
submission: /kaggle/working/submission.zip
